# Day 04 — File I/O, JSON, String Operations

============================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - The `with` statement — safe resource management
  - Reading and writing text files (.txt system prompts)
  - json.loads() and json.dumps() — parsing LLM JSON responses
  - Safe dict access with .get() — defensive parsing
  - String methods — cleaning, splitting, joining prompt text

WHY THIS MATTERS FOR LLM ENGINEERING:
  - System prompts live in .txt files (not hardcoded in code)
  - LLM responses come back as JSON strings — you MUST parse them
  - .get() is how you safely extract fields without crashing
  - String methods clean raw text before sending it to an LLM


In [ ]:

import json          # parse and produce JSON strings
import os            # file path operations
import csv           # read CSV datasets
from pathlib import Path  # cleaner file path handling than os.path


# Resolve the data directory relative to this file's location.
# This means the code works regardless of which directory you run it from.
# __file__ is the absolute path of the current .py file.
BASE_DIR   = Path(__file__).parent.parent   # module01/
DATA_DIR   = BASE_DIR / "data" / "datasets"
PROMPT_DIR = BASE_DIR / "prompts"




## SECTION 1: THE with STATEMENT — SAFE RESOURCE MANAGEMENT

═════════════════════════════════════════════════════════════════════════════
The with statement guarantees that a resource (file, DB connection, HTTP session)
is properly CLOSED even if an exception occurs inside the block.
WITHOUT with — dangerous:
f = open("file.txt")
data = f.read()          # if this raises, f.close() is never called!
f.close()
WITH the with statement — safe:
with open("file.txt") as f:
data = f.read()      # if this raises, Python still calls f.close()
Day 09 introduces async with — the same concept but for async resources.


In [ ]:

def load_system_prompt(filename: str = "system_prompt.txt") -> str:
    """
    Load a system prompt from a .txt file.

    Keeping system prompts in files (not in code) has important benefits:
    - Non-technical team members can edit prompts without touching code
    - Prompts can be version-controlled independently
    - Different prompts can be loaded based on environment (dev/prod)

    Args:
        filename: Name of the .txt file inside the prompts/ directory.

    Returns:
        The prompt as a string with leading/trailing whitespace removed.

    Raises:
        FileNotFoundError: If the prompt file does not exist.
    """

    filepath = PROMPT_DIR / filename

    # with open() as f: — the with statement
    # "r" = read mode (default)
    # encoding="utf-8" — always specify encoding to avoid platform differences
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # .strip() removes leading/trailing whitespace and newlines
    # This matters because text editors often add a trailing newline
    return content.strip()



In [ ]:
def save_llm_response_log(
    query: str,
    response: str,
    log_file: str = "llm_calls.log",
) -> None:
    """
    Append an LLM call record to a log file.

    Using "a" (append) mode means each call is added to the end
    of the file without overwriting existing records.

    Args:
        query    : The user query that was sent.
        response : The LLM response received.
        log_file : Name of the log file to append to.
    """

    log_path = BASE_DIR / log_file

    # "a" = append mode — creates the file if it doesn't exist
    with open(log_path, "a", encoding="utf-8") as f:
        # json.dumps() converts a Python dict to a JSON string
        # This makes log entries machine-readable (can be parsed later)
        log_entry = json.dumps({
            "query"   : query,
            "response": response,
        })
        f.write(log_entry + "\n")  # one JSON object per line (JSONL format)




## SECTION 2: json.loads() and json.dumps() — PARSING LLM RESPONSES

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def parse_llm_json_response(raw_response: str) -> dict:
    """
    Parse a JSON string returned by an LLM into a Python dict.

    LLMs are instructed to return JSON (Structured Output, Module 03 Technique 04).
    The raw response is always a STRING. You must parse it to get a dict.

    raw_response (str): '{"category": "URGENT", "confidence": "high"}'
    After json.loads:   {"category": "URGENT", "confidence": "high"}  (dict)

    Args:
        raw_response: The raw string returned by the LLM.

    Returns:
        A Python dict with the parsed JSON fields.

    Raises:
        json.JSONDecodeError: If the string is not valid JSON.
        ValueError: If the response contains markdown fences.
    """

    # LLMs sometimes wrap their JSON in markdown code fences:
    # ```json
    # {"key": "value"}
    # ```
    # Strip these before parsing.
    cleaned = raw_response.strip()
    if cleaned.startswith("```"):
        # Remove the opening fence (```json or ```)
        cleaned = cleaned.split("\n", 1)[-1]
        # Remove the closing fence
        cleaned = cleaned.rsplit("```", 1)[0]
        cleaned = cleaned.strip()

    # json.loads() parses a JSON string into a Python object
    # Raises json.JSONDecodeError if the string is not valid JSON
    parsed = json.loads(cleaned)

    return parsed



In [ ]:
def safe_get_field(
    data: dict,
    field: str,
    default: Any = None,     # type: ignore[name-defined]
    expected_type: type | None = None,
) -> Any:                    # type: ignore[name-defined]
    """
    Safely extract a field from a parsed LLM response dict.

    LLM responses are not guaranteed to contain every expected field.
    Always use .get() with a default value — never use data["field"]
    directly on LLM output (KeyError will crash your service).

    Args:
        data         : The parsed dict from the LLM.
        field        : The key to extract.
        default      : Value to return if the key is missing.
        expected_type: If provided, validate the type of the extracted value.

    Returns:
        The field value, or the default if missing.
    """

    # dict.get(key, default) — returns default if key is not present
    # This is always safer than data[key] on untrusted data
    value = data.get(field, default)

    # Optional type check
    if expected_type is not None and value is not None:
        if not isinstance(value, expected_type):
            # Log the type mismatch but don't crash — return default instead
            print(
                f"Warning: field '{field}' expected {expected_type.__name__}, "
                f"got {type(value).__name__}. Using default."
            )
            return default

    return value




## SECTION 3: READING CSV DATA — REAL E-COMMERCE DATASET

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def load_customers(limit: int = 10) -> list[dict]:
    """
    Load customer records from customers.csv.

    The csv module reads each row as a list of strings.
    DictReader maps column headers to values automatically.

    Args:
        limit: Maximum number of rows to load (use None for all rows).

    Returns:
        List of customer dicts, one per row.
    """

    filepath = DATA_DIR / "customers.csv"
    customers = []

    with open(filepath, "r", encoding="utf-8") as f:
        # csv.DictReader reads each row as a dict keyed by column headers
        reader = csv.DictReader(f)

        for i, row in enumerate(reader):
            if limit is not None and i >= limit:
                break   # stop after `limit` rows

            # Each row is already a dict: {"customer_id": "1001", "first_name": "Danielle", ...}
            # Values are strings from the CSV — convert types as needed
            customers.append({
                "customer_id": int(row["customer_id"]),
                "name"       : f"{row['first_name']} {row['last_name']}",
                "email"      : row["email"].lower().strip(),  # normalise email
                "city"       : row["city"],
                "state"      : row["state"],
                "country"    : row["country"],
            })

    return customers



In [ ]:
def load_few_shot_examples(filename: str = "few_shot_examples.json") -> list[dict]:
    """
    Load few-shot examples from a JSON file.

    Few-shot examples are stored in JSON so they can be edited without
    changing Python code. The file contains input/output pairs that
    are injected into the ## Examples section of the system prompt.

    Args:
        filename: Name of the JSON file inside prompts/.

    Returns:
        List of {"input": ..., "output": ...} dicts.
    """

    filepath = PROMPT_DIR / filename

    with open(filepath, "r", encoding="utf-8") as f:
        # json.load() reads from a file object (vs json.loads() which reads a string)
        data = json.load(f)

    # The JSON file has an "examples" key containing the list
    return data.get("examples", [])



In [ ]:
def format_few_shot_examples(examples: list[dict]) -> str:
    """
    Format few-shot examples into a string for the system prompt.

    Args:
        examples: List of {"input": ..., "output": ...} dicts.

    Returns:
        Formatted string ready to paste into the ## Examples section.
    """

    lines = []
    for ex in examples:
        lines.append(f"Input : {ex['input']}")
        lines.append(f"Output: {ex['output']}")
        lines.append("")   # blank line between examples

    return "\n".join(lines)




## SECTION 4: STRING OPERATIONS FOR PROMPT ENGINEERING

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def clean_product_description(raw_description: str, max_length: int = 300) -> str:
    """
    Clean and truncate a product description for use in a prompt.

    Raw text from a database often has:
    - Extra whitespace between words
    - Leading/trailing newlines
    - Tabs and other control characters

    Args:
        raw_description: The raw description string from the database.
        max_length      : Maximum characters to include in the prompt.

    Returns:
        A clean, truncated string ready for injection into a prompt.
    """

    # .strip() removes leading/trailing whitespace
    text = raw_description.strip()

    # .split() with no args splits on ANY whitespace (spaces, tabs, newlines)
    # and removes empty strings. " ".join() puts it back with single spaces.
    # This collapses multiple spaces/tabs/newlines into single spaces.
    words = text.split()
    text = " ".join(words)

    # Truncate if over the limit, adding "..." to signal truncation
    if len(text) > max_length:
        text = text[:max_length - 3] + "..."

    return text



In [ ]:
def build_product_context_for_prompt(product: dict) -> str:
    """
    Format a product dict into a structured context block for an LLM prompt.

    The context is wrapped in XML tags so the LLM knows this is
    retrieved data (not user input). This follows the prompt anatomy
    from Module 03: use <context> tags for RAG-retrieved content.

    Args:
        product: A dict with product fields.

    Returns:
        A formatted context string with XML tags.
    """

    # f-string with multiline format
    context = f"""<context>
Product ID   : {product.get('product_id', 'N/A')}
Product Name : {product.get('product_name', 'Unknown')}
Category     : {product.get('category', 'Unknown')}
Brand        : {product.get('brand', 'Unknown')}
Price        : ${product.get('price', 0):.2f}
In Stock     : {product.get('stock_quantity', 0)} units
Description  : {clean_product_description(product.get('description', ''))}
</context>"""

    return context




## Need Any for the safe_get_field function


In [ ]:
from typing import Any

if __name__ == "__main__":
    print("=" * 60)
    print("DAY 04 — File I/O, JSON, String Operations")
    print("=" * 60)

    # File I/O
    print("\n[1] Loading system prompt from file")
    try:
        prompt = load_system_prompt()
        print(f"  Loaded {len(prompt)} characters")
        print(f"  First 100 chars: {prompt[:100]}...")
    except FileNotFoundError as e:
        print(f"  File not found: {e}")

    # JSON parsing
    print("\n[2] Parsing LLM JSON response")
    mock_llm_response = '{"category": "URGENT", "confidence": "high", "reason": "Chest pain described"}'
    parsed = parse_llm_json_response(mock_llm_response)
    print(f"  Parsed type: {type(parsed)}")
    print(f"  Category  : {safe_get_field(parsed, 'category', default='UNKNOWN')}")
    print(f"  Missing   : {safe_get_field(parsed, 'missing_field', default='N/A')}")

    # LLM with markdown fence
    print("\n[3] Parsing LLM response with markdown fences")
    fenced = '```json\n{"intent": "TRACK_ORDER", "urgency": "high"}\n```'
    parsed2 = parse_llm_json_response(fenced)
    print(f"  Parsed: {parsed2}")

    # CSV loading
    print("\n[4] Loading customers from CSV")
    customers = load_customers(limit=3)
    for c in customers:
        print(f"  {c['customer_id']} | {c['name']:25s} | {c['city']}, {c['state']}")

    # Few-shot examples
    print("\n[5] Loading few-shot examples")
    try:
        examples = load_few_shot_examples()
        formatted = format_few_shot_examples(examples)
        print(formatted)
    except FileNotFoundError:
        print("  (few_shot_examples.json not found — expected in prompts/)")

    # String operations
    print("\n[6] Clean product description")
    raw = "  Interesting   help   church   successful   effort.   Care   base   know   goal.  "
    cleaned = clean_product_description(raw, max_length=50)
    print(f"  Raw    : '{raw[:40]}...'")
    print(f"  Cleaned: '{cleaned}'")


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day04_file_io_json.py', run_name='__main__')
